This is the notebook i used to train the 25M parameter model. Although I lost the model when I refreshed the page, the generated output is visible below

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# -------HYPERPARAMETERS------- #
batch_size = 64       
block_size = 512        
max_iters = 25000    
eval_interval = 500
learning_rate = 3e-4  
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 512         
n_head = 8            
n_layer = 8          
dropout = 0.2        
# ----------------------------- #

torch.manual_seed(3530)

print(f"Using device: {device}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

# -------IMPORTING THE DATA--------- #
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="Pranjal888/Pathology_textbook_training_dataset",
    filename="pathology_book_combined_data.txt",
    repo_type="dataset"
)

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Loaded {len(text):,} characters")
# ---------------------------------- #

# ------DATA CLEANING------- #
import re

def clean_text(text):
    allowed = set(
        "abcdefghijklmnopqrstuvwxyz"
        "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
        "0123456789"
        " \n\t"
        ".,;:!?()-–—'\"/°%+"
        "αβγδεζθκλμπστψω"
        "µ²³±≤≥"
    )
    return ''.join(c if c in allowed else ' ' for c in text)

text = clean_text(text)
text = re.sub(r' +', ' ', text)
text = re.sub(r'\n+', '\n', text)

chars = sorted(set(text))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
# ------------------------------- #

# ---------TOKENIZER---------- #
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
# ----------------------------- #

# -------------TRAIN TEST SPLIT-------------- #
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
# ------------------------------------------- #

# -----------DATA LOADING----------- #
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            if isinstance(loss, torch.Tensor) and loss.dim() > 0:
                loss = loss.mean()
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out
# ----------------------------------- #

# -----------MODEL ARCHITECTURE----------- #
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

class FeedFoward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx
# ----------------------------------------- #

# -------SETUP MODEL WITH DUAL GPU--------- #
base_model = BigramLanguageModel()

# wrap with DataParallel if 2 GPUs available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(base_model)
else:
    print("Using single GPU")
    model = base_model

model = model.to(device)
print(sum(p.numel() for p in base_model.parameters())/1e6, 'M parameters')
# ----------------------------------------- #

# -------OPTIMIZER--------- #
optimizer = torch.optim.AdamW(base_model.parameters(), lr=learning_rate)
# ------------------------- #

# -------TRAINING LOOP--------- #
for iter in range(max_iters):

    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)

    if isinstance(loss, torch.Tensor) and loss.dim() > 0:
        loss = loss.mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)
    optimizer.step()

    # save checkpoint every 1000 iters
    if iter % 1000 == 0:
        torch.save(
            base_model.state_dict(), 
            '/kaggle/working/pathology_gpt.pt'
        )
        print(f"checkpoint saved at iter {iter}")
# ------------------------------ #

# -------GENERATE TEXT--------- #
# use base_model for generation (not DataParallel wrapped)
base_model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(base_model.generate(context, max_new_tokens=2000)[0].tolist()))
# ------------------------------ #

Using device: cuda
Number of GPUs: 2
Loaded 52,746,593 characters
Vocabulary size: 103
Using 2 GPUs!
25.575527 M parameters
step 0: train loss 4.8265, val loss 4.8137
checkpoint saved at iter 0
step 500: train loss 2.2309, val loss 2.2263
step 1000: train loss 1.5465, val loss 1.5370
checkpoint saved at iter 1000
step 1500: train loss 1.3454, val loss 1.3529
step 2000: train loss 1.2368, val loss 1.2465
checkpoint saved at iter 2000
step 2500: train loss 1.1715, val loss 1.1839
step 3000: train loss 1.1320, val loss 1.1516
checkpoint saved at iter 3000
step 3500: train loss 1.0999, val loss 1.1189
step 4000: train loss 1.0745, val loss 1.0917
checkpoint saved at iter 4000
step 4500: train loss 1.0530, val loss 1.0684
step 5000: train loss 1.0338, val loss 1.0550
checkpoint saved at iter 5000
step 5500: train loss 1.0156, val loss 1.0386
step 6000: train loss 1.0029, val loss 1.0288
checkpoint saved at iter 6000
step 6500: train loss 0.9890, val loss 1.0159
step 7000: train loss 0.9779,